# Expense Baseline

This notebook builds a partial YTD expense baseline from the cleaned Quicken transaction export.

Quicken remains authoritative for non-Amazon expenses. Transfer rows, mortgage, State Farm, and Amazon payees are excluded. Amazon monthly category summaries are added in their place.

Mortgage and State Farm are modeled separately in the scenario rather than inferred here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analysis.amazon.notebook import load_monthly_summaries
from analysis.expenses.baseline import build_expense_baseline

quicken_path = Path('../data/shared/amazon/Transaction Export-2026-08-18.csv')
state_dir = Path('../state')

if not quicken_path.exists():
    raise FileNotFoundError(f'Quicken export not found at: {quicken_path.resolve()}')

amazon_monthly_summaries = load_monthly_summaries(state_dir)
baseline = build_expense_baseline(quicken_path, amazon_monthly_summaries)

print(f'Included Quicken expense rows: {len(baseline.transactions):,}')
print(f'Excluded Quicken rows: {len(baseline.excluded):,}')
print(f'Amazon summary rows: {len(amazon_monthly_summaries):,}')

## Category Totals

In [ ]:
baseline.category_totals

## Monthly Baseline

In [ ]:
monthly_baseline = (
    baseline.monthly.groupby(['summary_month_start', 'category'], as_index=False)['amount']
    .sum()
    .rename(columns={'amount': 'amount_total'})
    .sort_values(['summary_month_start', 'category'])
)
monthly_baseline

## Coverage and Exclusions

In [ ]:
excluded_summary = (
    baseline.excluded.assign(amount=lambda frame: frame['amount_cents'].abs() / 100)
    .groupby('exclude_reason', as_index=False)
    .agg(rows=('source_row', 'size'), amount=('amount', 'sum'))
    .sort_values('amount', ascending=False)
)
excluded_summary

In [ ]:
top_uncategorized = (
    baseline.transactions[baseline.transactions['category'].eq('Uncategorized')]
    .assign(amount=lambda frame: frame['amount_cents'].abs() / 100)
    .sort_values('amount', ascending=False)
    [['date', 'account', 'payee', 'amount', 'memo', 'source_row']]
    .head(25)
)
top_uncategorized

## Trend

In [ ]:
trend = monthly_baseline.pivot_table(
    index='summary_month_start',
    columns='category',
    values='amount_total',
    aggfunc='sum',
    fill_value=0.0,
)
top_categories = baseline.category_totals.head(8)['category'].tolist()
ax = trend[top_categories].plot(figsize=(11, 5), marker='o')
ax.set_title('Baseline Expense Categories')
ax.set_xlabel('Month')
ax.set_ylabel('Amount')
ax.legend(title='Category', loc='best')
plt.tight_layout()
plt.show()